In [1]:
# Celda 1 - Conexion y configuracion
import wrds
import pandas as pd
import numpy as np
import warnings

# La version de pandas del JupyterHub de WRDS emite falsos positivos de
# ChainedAssignmentError (incluso desde codigo interno de pandas).
# Se silencian; todas las columnas nuevas se crean con .assign()
warnings.filterwarnings('ignore', category=pd.errors.ChainedAssignmentError)

db = wrds.Connection(wrds_username='ppizam')

Loading library list...
Done


In [2]:
# Celda 2 - Vigencias permno-ticker desde crsp.stocknames
vidas = db.raw_sql("""
    select permno, permco, namedt, nameenddt,
           ticker, comnam, ncusip, cusip,
           shrcd, exchcd, siccd, shrcls
    from crsp.stocknames
    where nameenddt >= '2020-01-01'
      and namedt   <= '2024-12-31'
      and exchcd in (1, 2, 3, 4)
    order by ticker, namedt
""", date_cols=['namedt', 'nameenddt'])

print(len(vidas), vidas['permno'].nunique())
# Resultado esperado: 16828 11899

16828 11899


In [3]:
# Celda 3 - Delistings 2020-2024
bajas = db.raw_sql("""
    select permno, dlstdt, dlstcd, dlret
    from crsp.msedelist
    where dlstdt between '2020-01-01' and '2024-12-31'
""", date_cols=['dlstdt'])

print(len(bajas))

12903


In [5]:
# Celda 4 - Puente permno-gvkey (CCM no esta habilitado; se usa Financial Ratios Suite)
bridge = db.raw_sql("""
    select distinct permno, gvkey
    from wrdsapps_finratio.firm_ratio
    where public_date between '2020-01-01' and '2024-12-31'
""")
print(len(bridge), bridge['permno'].nunique())
# Resultado esperado: 5007 4816 (~190 permnos con 2+ gvkeys: reverse mergers)

multi = bridge[bridge.duplicated('permno', keep=False)]['permno'].unique()
bridge_1 = (bridge.sort_values(['permno', 'gvkey'])
                  .drop_duplicates('permno', keep='first'))

comp_info = db.raw_sql("""
    select gvkey, cik, gsector, ggroup, gind, conm
    from comp.company
""")
bridge_1 = bridge_1.merge(comp_info, on='gvkey', how='left')

vidas_enr = vidas.merge(bridge_1[['permno', 'gvkey', 'cik', 'gsector']],
                        on='permno', how='left')
vidas_enr = vidas_enr.assign(gvkey_multiple=vidas_enr['permno'].isin(multi))

print(len(vidas), len(vidas_enr))          # deben ser IGUALES: 16828 16828
print(round(vidas_enr['cik'].notna().mean(), 3))                                    # ~0.417 global
print(round(vidas_enr[vidas_enr['shrcd'].isin([10, 11])]['cik'].notna().mean(), 3)) # ~0.877 comunes


5007 4816
16828 16828
0.417
0.877


In [6]:
# Celda 5 - Consolidar tramos en vidas symbol-permno
# Robusta a los dtypes nullable de WRDS: sin el fillna y el astype('bool'),
# groupby descarta llaves NA en silencio y se pierden 8,500 permnos
W0, W1 = pd.Timestamp('2020-01-01'), pd.Timestamp('2024-12-31')

v = vidas_enr.copy().sort_values(['permno', 'namedt'])

prev_ticker = v.groupby('permno')['ticker'].shift()
prev_fin    = v.groupby('permno')['nameenddt'].shift()

cambio = (
    prev_ticker.isna()                                               # primer tramo del permno
    | (v['ticker'] != prev_ticker).fillna(False)                     # cambio de ticker
    | (v['namedt'] > prev_fin + pd.Timedelta(days=7)).fillna(False)  # hueco > 7 dias
)
cambio = cambio.astype('bool')

v = v.assign(
    fecha_inicio=v['namedt'].clip(lower=W0),
    fecha_fin=v['nameenddt'].clip(upper=W1),
    bloque=cambio.groupby(v['permno']).cumsum().astype(int),
)

vig = v.groupby(['permno', 'bloque'], as_index=False).agg(
    ticker=('ticker', 'last'), comnam=('comnam', 'last'),
    fecha_inicio=('fecha_inicio', 'min'), fecha_fin=('fecha_fin', 'max'),
    ncusip=('ncusip', 'last'), shrcd=('shrcd', 'last'),
    exchcd=('exchcd', 'last'), siccd=('siccd', 'last'),
    permco=('permco', 'last'), gvkey=('gvkey', 'last'),
    cik=('cik', 'last'), gsector=('gsector', 'last'),
    gvkey_multiple=('gvkey_multiple', 'max'),
)

print(len(v), 'tramos ->', len(vig), 'vidas |',
      v['permno'].nunique(), 'permnos en origen,', vig['permno'].nunique(), 'en vig')
assert vig['permno'].nunique() == v['permno'].nunique(), 'se perdieron permnos'
# Resultado esperado: 16828 tramos -> 12961 vidas | 11899 permnos en ambos

16828 tramos -> 12961 vidas | 11899 permnos en origen, 11899 en vig


In [7]:
# Celda 6 - Estados, renombres, tickers reutilizados, ipo_year y estado_vida

# Delistings: solo el ultimo tramo del permno y solo dlstcd >= 200 (100 = activo)
bajas_u = bajas.sort_values('dlstdt').drop_duplicates('permno', keep='last')
vig = vig.merge(bajas_u[['permno', 'dlstdt', 'dlstcd']], on='permno', how='left')

es_ultima = vig['fecha_fin'] == vig.groupby('permno')['fecha_fin'].transform('max')
es_baja = es_ultima & vig['dlstdt'].notna() & (vig['dlstcd'] >= 200)
vig = vig.assign(
    listing_status=np.where(es_baja, 'delisted', 'active'),
    delist_date=vig['dlstdt'].where(es_baja),
)

# Cambios de ticker: baja + alta enlazadas por permno
vig = vig.sort_values(['permno', 'fecha_inicio'])
vig = vig.assign(
    ticker_previo=vig.groupby('permno')['ticker'].shift(),
    ticker_siguiente=vig.groupby('permno')['ticker'].shift(-1),
)

# Tickers usados por mas de un permno en la ventana
vig = vig.assign(
    ticker_reutilizado=vig['ticker'].map(vig.groupby('ticker')['permno'].nunique()) > 1
)

# ipo_year: primera aparicion del permno en TODA la historia de CRSP
primeras = db.raw_sql("""
    select permno, min(namedt) as primera_fecha
    from crsp.stocknames
    group by permno
""", date_cols=['primera_fecha'])
vig = vig.merge(primeras, on='permno', how='left')
vig = vig.assign(ipo_year=vig['primera_fecha'].dt.year)

# Estado de la vida
vig = vig.assign(
    estado_vida=np.select(
        [vig['listing_status'].eq('delisted'), vig['ticker_siguiente'].notna()],
        ['delisted', 'renombrada'], default='active',
    )
)
print(vig['estado_vida'].value_counts())
# Resultado esperado: active 8900 | delisted 2999 | renombrada 1062

estado_vida
active        8900
delisted      2999
renombrada    1062
Name: count, dtype: int64


In [8]:
 # Celda 7 - Checklist de anclas (control de calidad obligatorio)
anclas = ['GME', 'AMC', 'BBBY', 'FB', 'META', 'ABNB', 'COIN', 'HOOD',
          'RIVN', 'ARM', 'RDDT', 'DJT', 'SMCI', 'SIVB', 'SBNY']
cols = ['ticker', 'permno', 'comnam', 'fecha_inicio', 'fecha_fin',
        'listing_status', 'delist_date', 'ticker_previo', 'ticker_siguiente']
print(vig[vig['ticker'].isin(anclas)][cols]
      .sort_values(['ticker', 'fecha_inicio']).to_string(index=False))
# Verificar: GME desde 2020-01-01; FB->META (permno 13407); META tambien como
# ETF 2021-2022 (permno 21413, sucesor METV); DJT con previo DWAC;
# SMCI desde 2020-01-14; BBBY delisted 2023-05-02; SIVB/SBNY 2023-03-10

ticker  permno                           comnam fecha_inicio  fecha_fin listing_status delist_date ticker_previo ticker_siguiente
  ABNB   20190                       AIRBNB INC   2020-12-10 2024-12-31         active         NaT          <NA>             <NA>
   AMC   14328 A M C ENTERTAINMENT HOLDINGS INC   2020-01-01 2024-12-31         active         NaT          <NA>             <NA>
   ARM   24252               A R M HOLDINGS PLC   2023-09-14 2024-12-31         active         NaT          <NA>             <NA>
  BBBY   77659            BED BATH & BEYOND INC   2020-01-01 2023-05-02       delisted  2023-05-02          <NA>             <NA>
  COIN   20892              COINBASE GLOBAL INC   2021-04-14 2024-12-31         active         NaT          <NA>             <NA>
   DJT   21927      TRUMP MEDIA & TECH GRP CORP   2024-03-26 2024-12-31         active         NaT          DWAC             <NA>
    FB   13407               META PLATFORMS INC   2020-01-01 2022-06-08         active    

In [9]:
# Celda 8 - Respaldo: csv base sin campos de mercado
vig.to_csv('padron_vigencias_2020_2024.csv', index=False)

In [10]:
# Celda 9 - Archivo mensual de CRSP y market cap
msf = db.raw_sql("""
    select permno, date, abs(prc) as precio,
           shrout, vol, ret
    from crsp.msf
    where date between '2020-01-01' and '2024-12-31'
""", date_cols=['date'])

msf = msf.assign(market_cap_usd=msf['precio'] * msf['shrout'] * 1000)
print(len(msf), msf['permno'].nunique())

546882 12903


In [11]:
# Celda 10 - Snapshot de mercado por vida (no por permno)
# FB toma su ultimo mes de vida (may-2022), no el de META

# Verificado con GME ene-2021: vol de crsp.msf viene en CENTENAS de acciones
# (CRSP reporto 12,623,971 vs ~1,262 millones de acciones reales del mes)
FACTOR_VOL = 100

m = msf.merge(vig[['permno', 'bloque', 'fecha_inicio', 'fecha_fin']],
              on='permno', how='inner')
m = m[(m['date'] >= m['fecha_inicio']) & (m['date'] <= m['fecha_fin'])]
m = m.assign(dollar_vol_mes=m['precio'] * m['vol'] * FACTOR_VOL)

# Ultimo mes de cada vida = snapshot
snap = (m.sort_values('date')
          .groupby(['permno', 'bloque'], as_index=False).tail(1)
          .rename(columns={'date': 'snapshot_mercado',
                           'precio': 'last_price_usd',
                           'shrout': 'shares_outstanding_miles'}))

# Promedio de volumen en dolares sobre todos los meses de la vida
adv = (m.groupby(['permno', 'bloque'], as_index=False)
         .agg(avg_dollar_volume_usd_mensual=('dollar_vol_mes', 'mean'),
              meses_con_datos=('dollar_vol_mes', 'size')))

vig = vig.merge(snap[['permno', 'bloque', 'snapshot_mercado', 'last_price_usd',
                      'shares_outstanding_miles', 'market_cap_usd']],
                on=['permno', 'bloque'], how='left')
vig = vig.merge(adv, on=['permno', 'bloque'], how='left')

# Aproximacion diaria (~21 dias habiles por mes) para el piso de liquidez
vig = vig.assign(avg_dollar_volume_usd_diario=vig['avg_dollar_volume_usd_mensual'] / 21)

print(round(vig['last_price_usd'].notna().mean(), 4))
# Resultado esperado: ~0.997 (las ~39 vidas sin datos son units/warrants/vidas muy cortas)

0.997


In [12]:
# Celda 11 - Sanity checks del snapshot
print(vig[vig['ticker'] == 'GME']
      [['ticker', 'snapshot_mercado', 'last_price_usd', 'market_cap_usd',
        'avg_dollar_volume_usd_diario']].to_string(index=False))
# Esperado: snapshot 2024-12-31, precio 31.34, mcap ~14,000 millones

print(vig[(vig['ticker'] == 'FB') & (vig['permno'] == 13407)]
      [['ticker', 'fecha_fin', 'snapshot_mercado', 'last_price_usd']].to_string(index=False))
# Esperado: snapshot 2022-05-31, precio 193.64 (por vida, no por permno)

assert len(vig) == 12961, 'el merge duplico filas'

ticker snapshot_mercado  last_price_usd  market_cap_usd  avg_dollar_volume_usd_diario
   GME       2024-12-31           31.34   14002712000.0              813515092.374959
ticker  fecha_fin snapshot_mercado  last_price_usd
    FB 2022-06-08       2022-05-31          193.64


In [13]:
# Celda 12 - Exportacion final con candado
assert 'market_cap_usd' in vig.columns, 'vig no trae las columnas de mercado; corre las celdas 9 y 10'
vig.to_csv('padron_vigencias_2020_2024_mkt.csv', index=False)
print(len(vig.columns), 'columnas exportadas')   # deben ser ~32, no 25

32 columnas exportadas
